In [71]:
pip install pypdf numpy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Cell 1 - install
!pip install PyPDF2 tiktoken google-generativeai

# Cell 2 - set key (never screenshot this cell)
import os
os.environ["GEMINI_API_KEY"] = "your_api_key_here"

In [73]:
!pip install google-genai

In [74]:
!pip install chromadb

In [75]:
import os
import math
import tiktoken
import PyPDF2
from google import genai
from google.genai import types
import chromadb

file_path= "/Users/azeemkhalipha/ai-engineer-portfolio/Attention_Is_All_You_Need.pdf"


# load pdf document and extract text
def load_pdf(file_path:str) ->str:
    read_pdf = PyPDF2.PdfReader(file_path)
    text = ""
    for page in read_pdf.pages:
        text += page.extract_text() or "" 
    return text





In [76]:
# chunk text into smaller pieces for embedding
# window_size is the max number of tokens per chunk, overlap is the number of tokens that overlap between chunks to preserve context
# returns a list of text chunks
# you can adjust window_size and overlap based on the embedding model's token limit and how much context you want to preserve between chunks
def chunk_text(text: str, window_size: int = 512, overlap: int = 50) -> list:
    tokenizer = tiktoken.get_encoding("cl100k_base")
    tokens = tokenizer.encode(text)  # converts string → list of token IDs
    chunks= []
    for i in range(0, len(tokens), window_size - overlap):
        chunk = tokens[i:i + window_size]
        chunks.append(tokenizer.decode(chunk))  # converts list of token IDs → string
    return chunks



In [77]:
# embed chunks using Gemini embedding model and store in chromadb
# returns a list of dictionaries with chunk_id, text, and embedding
# you can adjust the batch_size based on the embedding model's rate limits and your system's memory constraints
def embed_chunks(chunks: list, client) -> list:
    store = []
    batch_size = 100
    for i in range(0, len(chunks), batch_size):
        batch= chunks[i:i+batch_size]
        response= client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=batch,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"))
        for j, embedding in enumerate(response.embeddings): 
            store.append({"chunk_id":i+j,
                          "text": batch[j],
                          "embedding": embedding.values})
    return store



# embed chunks and store in chromadb collection
# you can adjust the batch_size based on the embedding model's rate limits and your system's memory constraints
def embed_and_store(texts: list[str], client, collection: chromadb.Collection):
    batch_size = 100
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.models.embed_content(model="models/gemini-embedding-001", contents=batch, config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"))
        embeddings = [emb.values for emb in response.embeddings]
        ids = [f"chunk_{i + j}" for j in range(len(batch))]
        collection.add(documents=batch, embeddings=embeddings, ids=ids)


# search chromadb collection using query embedding and return results
# you can adjust the top_k parameter to return more or fewer results based on your needs
def search_chromadb(query: str, client, collection: chromadb.Collection, top_k: int = 3) -> list[str]:
    response = client.models.embed_content(model="models/gemini-embedding-001", contents=[query], config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"))
    query_embedding = response.embeddings[0].values
    results = collection.query(query_embeddings=[query_embedding], n_results=top_k)
    return results

In [78]:
# compute cosine similarity between two vectors
# returns a value between -1 and 1 where 1 means the vectors are identical, 0 means they are orthogonal, and -1 means they are opposite
def cosine_similarity(vec_a: list, vec_b: list) -> float:
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    magnitude_a = math.sqrt(sum(a ** 2 for a in vec_a))
    magnitude_b = math.sqrt(sum(b ** 2 for b in vec_b))
    if magnitude_a == 0 or magnitude_b == 0:
        return 0.0
    return dot_product / (magnitude_a * magnitude_b)

In [79]:
# search chunks using query embedding and cosine similarity
# returns a list of dictionaries with chunk_id, text, and similarity score sorted by similarity in descending order
def search_chunks(query: str, store: list, client, top_k: int = 3) -> list:
    query_response = client.models.embed_content(
        model="models/gemini-embedding-001",
        contents=query,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY"
        )
    )
    query_embedding = query_response.embeddings[0].values

    results = []
    for item in store:
        similarity = cosine_similarity(query_embedding, item["embedding"])
        results.append({"chunk_id": item["chunk_id"], "text": item["text"], "similarity": similarity})

    # Sort results by similarity in descending order and return top_k
    results.sort(key=lambda x: x["similarity"], reverse=True)
    return results[:top_k]

In [80]:
# example usage
if __name__ == "__main__":
    client= genai.Client()
    chroma_client = chromadb.PersistentClient(path="./chroma_db")
    collection = chroma_client.get_or_create_collection(name="pdf_chunks")
    text= load_pdf(file_path)
    chunks= chunk_text(text)
    if collection.count() == 0:
        embed_and_store(chunks, client, collection)
    else:
        print(f"Collection already has {collection.count()} chunks. Skipping embedding.")
    query= "What is the transformer architecture?"
    results= search_chromadb(query, client, collection)
    
    for doc, distance, chunk_id in zip(results['documents'][0], results['distances'][0], results['ids'][0]):
        print(f"\n[Chunk ID: {chunk_id} | Similarity Score: {distance:.4f}]")
        print(doc.strip())

Collection already has 22 chunks. Skipping embedding.

[Chunk ID: chunk_3 | Similarity Score: 0.4870]
18] and [9].
3 Model Architecture
Most competitive neural sequence transduction models have an encoder-decoder structure [ 5,2,35].
Here, the encoder maps an input sequence of symbol representations (x1, ..., x n)to a sequence
of continuous representations z= (z1, ..., z n). Given z, the decoder then generates an output
sequence (y1, ..., y m)of symbols one element at a time. At each step the model is auto-regressive
[10], consuming the previously generated symbols as additional input when generating the next.
2Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N= 6 identical layers. Each layer has two